In [1]:
import numpy as np
import pandas as pd

In [32]:
data = pd.read_csv("/content/data_for_model_building.csv")
data.head()

,text,label
0,mike traumatic season parent hospital little s...,-1
1,"bad yall. wth writing, pacing volume bad too. ...",-1
2,not single shot final episode,0
3,rated j jesus box office blasphemy hollywood c...,0
4,hollyweird ever stop attacking christianity. b...,-1


In [33]:
data.shape

(19596, 2)

In [41]:
na_indices = data[data["text"].isna()].index.to_list()

In [42]:
len(na_indices)

425

In [43]:
data.dropna(inplace=True)

In [44]:
data.shape

(19171, 2)

In [45]:
X = data.loc[:, "text"]
y = data.loc[:, "label"]

In [52]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [55]:
# tfidf vectorizer with max_features =14000
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=14000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [56]:
# label encoding the target variable because XgBoost cant deal iwth -1
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [58]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [59]:
xgboost = XGBClassifier(
    n_estimators=384,
    max_depth=9,
    learning_rate=0.14432103031505297,
    subsample=0.9636819545620402,
    colsample_bytree=0.7665904431247078,
    gamma=1.77936321036146,
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgboost.fit(X_train_tfidf, y_train_encoded)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7665904431247078, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=1.77936321036146, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.14432103031505297,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=384, n_jobs=-1, num_class=3, ...)

In [62]:
y_pred= xgboost.predict(X_test_tfidf)

xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test_encoded, y_pred),
    "Precision (macro)": precision_score(y_test_encoded, y_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_encoded, y_pred, average="macro", zero_division=0),
    "F1 Score (macro)": f1_score(y_test_encoded, y_pred, average="macro", zero_division=0),
}

In [63]:
results_df_xgb = pd.DataFrame([xgb_results])
results_df_xgb

,Model,Accuracy,Precision (macro),Recall (macro),F1 Score (macro)
0,XGBoost,0.707692,0.721457,0.696161,0.702105
